# Pubmed Dataset Preparation for GWM-RNN (Balanced Negative Mining)

Prepare Pubmed dataset for **GWM-RNN link prediction** with balanced negative sampling.

**Task:** Given two papers, predict if there's a citation link between them.

**GWM-RNN Data Format:**
- Sequence: `[Self_u, Context_u, Self_v, Context_v]`
- Shape: `[4, embedding_dim]` per edge
- Self: Node's own embedding
- Context: Mean of 1-hop neighbors

**Negative Sampling Strategy:**
- ✅ **Balanced**: 50% hard negatives (2-hop) + 50% random negatives
- ✅ Better recall/precision balance
- ✅ Fewer false negatives

---

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch sentence-transformers torch-geometric torch-scatter torch-sparse

print("✓ Dependencies installed successfully")

## 2. Import Libraries

In [ ]:
import os
import json
import shutil
import torch
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from torch_geometric.utils import k_hop_subgraph
from huggingface_hub import hf_hub_download
from tqdm import tqdm
from collections import defaultdict

# Check environment
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Configuration

In [ ]:
# Configuration
CONFIG = {
    'output_dir': '/kaggle/working/pubmed_gwm_rnn_data',
    'encoder_model': 'sentence-transformers/all-MiniLM-L6-v2',  # Lightweight and fast
    'embedding_dim': 384,  # Native dimension for all-MiniLM-L6-v2
    'context_hops': 1,  # Number of hops for context aggregation (1, 2, 3, etc.)
    'neg_sampling_ratio': 1.0,  # 1:1 positive to negative ratio
    'hard_negative_ratio': 0.5,  # BALANCED: 50% hard, 50% random
    'val_size': 0.10,  # 10% validation split
    'test_size': 0.10,  # 10% test split (80% training)
    'random_state': 42,
    'batch_size': 64,  # Encoding batch size
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CONFIG['device'] = device

print("="*70)
print(" "*20 + "GWM-RNN Data Processing Configuration")
print("="*70)
print("\nConfiguration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

print(f"\n📊 Data Format:")
print(f"  Sequence: [Self_u, Context_u, Self_v, Context_v]")
print(f"  Context: Mean of {CONFIG['context_hops']}-hop neighbors")
print(f"  Context: Mean of 1-hop neighbors")

print(f"\n⚖️  Negative Sampling:")
print(f"  {CONFIG['hard_negative_ratio']*100:.0f}% hard (2-hop neighbors) + {(1-CONFIG['hard_negative_ratio'])*100:.0f}% random")
print(f"  Expected: Balanced recall/precision")

## 4. Download Raw Pubmed Dataset

In [ ]:
print("="*70)
print(" "*20 + "STEP 1: Downloading Raw Data")
print("="*70)

repo_id = "Graph-COM/Text-Attributed-Graphs"
filename = "pubmed/processed_data.pt"
raw_dir = Path("/kaggle/temp/pubmed_raw")
raw_dir.mkdir(parents=True, exist_ok=True)
dest_file = raw_dir / "data.pt"

if dest_file.exists():
    print(f"✓ Raw data already exists at {dest_file}")
else:
    print(f"Downloading from {repo_id}...")
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type="dataset"
    )
    
    print(f"Copying to {dest_file}...")
    shutil.copy(downloaded_path, dest_file)

# Load and verify
print("\nLoading dataset...")
data = torch.load(dest_file, weights_only=False)

print(f"\n✓ Successfully loaded Pubmed dataset!")
print(f"  Nodes: {data.num_nodes:,}")
print(f"  Edges: {data.edge_index.size(1):,}")
print(f"  Classes: {len(data.label_texts)}")
print(f"  Has text: {hasattr(data, 'raw_texts')}")

if not hasattr(data, 'raw_texts'):
    raise ValueError("Dataset missing 'raw_texts' attribute!")

print(f"\nExample text (first 150 chars):")
print(f"  {data.raw_texts[0][:150]}...")

## 5. Generate Node Embeddings

Encode all node texts using sentence-transformers (frozen encoder).

In [ ]:
print("="*70)
print(" "*20 + "STEP 2: Generating Node Embeddings")
print("="*70)

print(f"Loading encoder: {CONFIG['encoder_model']}")
encoder = SentenceTransformer(CONFIG['encoder_model'], device=device)
encoder.eval()

print(f"✓ Encoder loaded on {device}")
print(f"  Output dimension: {encoder.get_sentence_embedding_dimension()}")

# Generate embeddings (frozen encoder, no fine-tuning)
print(f"\nEncoding {len(data.raw_texts):,} texts (batch_size={CONFIG['batch_size']})...")
with torch.no_grad():
    node_embeddings = encoder.encode(
        data.raw_texts,
        batch_size=CONFIG['batch_size'],
        show_progress_bar=True,
        convert_to_tensor=True,
        device=device
    )

# Move to CPU for processing
node_embeddings = node_embeddings.cpu()

print(f"\n✓ Generated embeddings shape: {node_embeddings.shape}")
print(f"  Expected dimension: {CONFIG['embedding_dim']}")

# Verify dimension
actual_dim = node_embeddings.shape[1]
if actual_dim != CONFIG['embedding_dim']:
    print(f"⚠️  Updating embedding_dim: {CONFIG['embedding_dim']} → {actual_dim}")
    CONFIG['embedding_dim'] = actual_dim

# Free GPU memory
del encoder
if device == 'cuda':
    torch.cuda.empty_cache()
    print(f"✓ Freed GPU memory")

## 6. Compute Multi-Hop Context Embeddings

For each node, compute the mean embedding of its multi-hop neighbors.

In [ ]:
print("="*70)
print(" "*20 + f"STEP 3: Computing {CONFIG['context_hops']}-Hop Context Embeddings")
print("="*70)

num_nodes = data.num_nodes
embedding_dim = node_embeddings.shape[1]
context_hops = CONFIG['context_hops']

# Compute context embeddings (mean of k-hop neighbors)
print(f"\nComputing {context_hops}-hop context embeddings for {num_nodes:,} nodes...")
print(f"Strategy: Aggregate all neighbors within {context_hops} hop(s)")

context_embeddings = torch.zeros(num_nodes, embedding_dim)

nodes_with_neighbors = 0
nodes_without_neighbors = 0
total_neighbors_aggregated = 0

for node_idx in tqdm(range(num_nodes), desc=f"Computing {context_hops}-hop contexts"):
    # Get k-hop neighbors using PyTorch Geometric's k_hop_subgraph
    subset, _, _, _ = k_hop_subgraph(
        node_idx=node_idx,
        num_hops=context_hops,
        edge_index=data.edge_index,
        relabel_nodes=False,
        num_nodes=num_nodes
    )
    
    # Exclude the center node itself
    neighbors = [n for n in subset.tolist() if n != node_idx]
    
    if len(neighbors) > 0:
        # Mean pooling of neighbor embeddings
        neighbor_embs = node_embeddings[neighbors]
        context_embeddings[node_idx] = neighbor_embs.mean(dim=0)
        nodes_with_neighbors += 1
        total_neighbors_aggregated += len(neighbors)
    else:
        # No neighbors: use node's own embedding as context
        context_embeddings[node_idx] = node_embeddings[node_idx]
        nodes_without_neighbors += 1

avg_neighbors = total_neighbors_aggregated / nodes_with_neighbors if nodes_with_neighbors > 0 else 0

print(f"\n✓ Context embeddings computed:")
print(f"  Nodes with neighbors: {nodes_with_neighbors:,}")
print(f"  Nodes without neighbors: {nodes_without_neighbors:,}")
print(f"  Average neighbors per node: {avg_neighbors:.1f}")
print(f"  Shape: {context_embeddings.shape}")

## 7. Split Edges and Generate Negative Samples

Split edges and generate balanced negative samples (50% hard + 50% random).

In [ ]:
print("="*70)
print(" "*20 + "STEP 4: Edge Split & Negative Sampling")
print("="*70)

# Extract edges
edge_index = data.edge_index
num_edges = edge_index.size(1)
positive_edges = edge_index.t().numpy()

print(f"Total edges: {num_edges:,}")

# Split edges into train/val/test
print("\nSplitting edges...")
train_val_edges, test_edges = train_test_split(
    positive_edges,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['random_state']
)

val_size_adjusted = CONFIG['val_size'] / (1 - CONFIG['test_size'])
train_edges, val_edges = train_test_split(
    train_val_edges,
    test_size=val_size_adjusted,
    random_state=CONFIG['random_state']
)

print(f"✓ Edge split:")
print(f"  Train: {len(train_edges):,} ({len(train_edges)/num_edges*100:.1f}%)")
print(f"  Val: {len(val_edges):,} ({len(val_edges)/num_edges*100:.1f}%)")
print(f"  Test: {len(test_edges):,} ({len(test_edges)/num_edges*100:.1f}%)")

# Build full adjacency for negative sampling
print("\nBuilding adjacency structure for negative mining...")
full_adj_list = defaultdict(set)
for src, dst in positive_edges:
    full_adj_list[src].add(dst)
    full_adj_list[dst].add(src)  # Undirected

print(f"✓ Adjacency built for {len(full_adj_list)} nodes")

# Negative sampling functions
def get_2hop_neighbors_only(node_idx, edge_index, num_nodes):
    """Get only 2-hop neighbors (excluding direct neighbors) using PyG's k_hop_subgraph."""
    # Convert to tensor if needed
    if not isinstance(node_idx, torch.Tensor):
        node_idx = torch.tensor([node_idx], dtype=torch.long)
    
    # Get all nodes within 2 hops
    subset_2hop, _, _, _ = k_hop_subgraph(
        node_idx=node_idx,
        num_hops=2,
        edge_index=edge_index,
        relabel_nodes=False,
        num_nodes=num_nodes
    )
    
    # Get all nodes within 1 hop (direct neighbors)
    subset_1hop, _, _, _ = k_hop_subgraph(
        node_idx=node_idx,
        num_hops=1,
        edge_index=edge_index,
        relabel_nodes=False,
        num_nodes=num_nodes
    )
    
    # Get only 2-hop neighbors (exclude center node and direct neighbors)
    two_hop_only = set(subset_2hop.tolist()) - set(subset_1hop.tolist())
    # node_idx might be a tensor with one element, so extract the value
    center_node = node_idx.item() if isinstance(node_idx, torch.Tensor) else node_idx
    two_hop_only.discard(center_node)
    
    return two_hop_only

def generate_hard_negative_edges(positive_edges, edge_index, num_nodes, num_negative, random_state=42):
    """Generate hard negative edges from 2-hop neighbors."""
    np.random.seed(random_state)
    positive_set = set(map(tuple, positive_edges))
    negative_edges = []
    
    nodes_with_edges = list(set(positive_edges[:, 0]) | set(positive_edges[:, 1]))
    
    max_attempts = num_negative * 20
    attempts = 0
    hard_found = 0
    
    with tqdm(total=num_negative, desc="Mining hard negatives") as pbar:
        while len(negative_edges) < num_negative and attempts < max_attempts:
            src = np.random.choice(nodes_with_edges)
            two_hop_neighbors = get_2hop_neighbors_only(src, edge_index, num_nodes)
            
            if len(two_hop_neighbors) > 0:
                dst = np.random.choice(list(two_hop_neighbors))
                hard_found += 1
            else:
                dst = np.random.randint(0, num_nodes)
            
            if src != dst and (src, dst) not in positive_set:
                negative_edges.append([src, dst])
                pbar.update(1)
            
            attempts += 1
    
    print(f"  Hard negatives: {hard_found}/{len(negative_edges)}")
    return np.array(negative_edges)

def generate_random_negative_edges(positive_edges, num_nodes, num_negative, random_state=42):
    """Generate random negative edges."""
    np.random.seed(random_state)
    positive_set = set(map(tuple, positive_edges))
    negative_edges = []
    
    max_attempts = num_negative * 10
    attempts = 0
    
    with tqdm(total=num_negative, desc="Generating random negatives") as pbar:
        while len(negative_edges) < num_negative and attempts < max_attempts:
            src = np.random.randint(0, num_nodes)
            dst = np.random.randint(0, num_nodes)
            
            if src != dst and (src, dst) not in positive_set:
                negative_edges.append([src, dst])
                pbar.update(1)
            
            attempts += 1
    
    return np.array(negative_edges)

def generate_mixed_negative_edges(positive_edges, edge_index, num_nodes, num_negative, hard_ratio=0.5, random_state=42):
    """Generate mixed negatives: hard + random."""
    num_hard = int(num_negative * hard_ratio)
    num_random = num_negative - num_hard
    
    print(f"  Target: {num_hard:,} hard + {num_random:,} random = {num_negative:,} total")
    
    hard_negs = generate_hard_negative_edges(positive_edges, edge_index, num_nodes, num_hard, random_state=random_state)
    random_negs = generate_random_negative_edges(positive_edges, num_nodes, num_random, random_state=random_state + 1000)
    
    mixed_negs = np.vstack([hard_negs, random_negs])
    np.random.seed(random_state)
    np.random.shuffle(mixed_negs)
    
    return mixed_negs

# Generate mixed negatives for each split
print(f"\n{'='*70}")
print("Generating mixed negative samples...")
print(f"Strategy: {CONFIG['hard_negative_ratio']*100:.0f}% hard + {(1-CONFIG['hard_negative_ratio'])*100:.0f}% random")
print(f"{'='*70}")

num_nodes = data.num_nodes
num_train_neg = int(len(train_edges) * CONFIG['neg_sampling_ratio'])
num_val_neg = int(len(val_edges) * CONFIG['neg_sampling_ratio'])
num_test_neg = int(len(test_edges) * CONFIG['neg_sampling_ratio'])

print(f"\n[TRAIN] Generating {num_train_neg:,} negatives...")
train_neg_edges = generate_mixed_negative_edges(
    positive_edges, data.edge_index, num_nodes, num_train_neg,
    hard_ratio=CONFIG['hard_negative_ratio'],
    random_state=CONFIG['random_state']
)

print(f"\n[VAL] Generating {num_val_neg:,} negatives...")
val_neg_edges = generate_mixed_negative_edges(
    positive_edges, data.edge_index, num_nodes, num_val_neg,
    hard_ratio=CONFIG['hard_negative_ratio'],
    random_state=CONFIG['random_state'] + 1
)

print(f"\n[TEST] Generating {num_test_neg:,} negatives...")
test_neg_edges = generate_mixed_negative_edges(
    positive_edges, data.edge_index, num_nodes, num_test_neg,
    hard_ratio=CONFIG['hard_negative_ratio'],
    random_state=CONFIG['random_state'] + 2
)

print(f"\n✓ Negative samples generated:")
print(f"  Train: {len(train_neg_edges):,}")
print(f"  Val: {len(val_neg_edges):,}")
print(f"  Test: {len(test_neg_edges):,}")

## 8. Create GWM-RNN Sequences

Create sequences in format: `[Self_u, Context_u, Self_v, Context_v]`

In [ ]:
print("="*70)
print(" "*20 + "STEP 5: Creating GWM-RNN Sequences")
print("="*70)

def create_sequences(pos_edges, neg_edges, node_embs, context_embs):
    """
    Create sequences: [Self_u, Context_u, Self_v, Context_v]
    
    Args:
        pos_edges: Positive edges (N, 2)
        neg_edges: Negative edges (M, 2)
        node_embs: Node embeddings (num_nodes, dim)
        context_embs: Context embeddings (num_nodes, dim)
    
    Returns:
        sequences: (N+M, 4, dim)
        labels: (N+M,) - 1 for positive, 0 for negative
    """
    all_edges = np.vstack([pos_edges, neg_edges])
    labels = np.concatenate([
        np.ones(len(pos_edges), dtype=np.int64),
        np.zeros(len(neg_edges), dtype=np.int64)
    ])
    
    sequences = []
    for src, dst in tqdm(all_edges, desc="Creating sequences"):
        sequence = torch.stack([
            node_embs[src],       # Self_u
            context_embs[src],    # Context_u
            node_embs[dst],       # Self_v
            context_embs[dst]     # Context_v
        ])  # Shape: [4, embedding_dim]
        sequences.append(sequence)
    
    sequences = torch.stack(sequences)  # Shape: [num_samples, 4, embedding_dim]
    labels = torch.from_numpy(labels)
    
    return sequences, labels

print("Creating training sequences...")
train_sequences, train_labels = create_sequences(
    train_edges, train_neg_edges, node_embeddings, context_embeddings
)

print("Creating validation sequences...")
val_sequences, val_labels = create_sequences(
    val_edges, val_neg_edges, node_embeddings, context_embeddings
)

print("Creating test sequences...")
test_sequences, test_labels = create_sequences(
    test_edges, test_neg_edges, node_embeddings, context_embeddings
)

print(f"\n✓ Sequences created:")
print(f"  Train: {train_sequences.shape} (labels: {train_labels.shape})")
print(f"  Val: {val_sequences.shape} (labels: {val_labels.shape})")
print(f"  Test: {test_sequences.shape} (labels: {test_labels.shape})")

print(f"\nSequence format: [Self_u, Context_u, Self_v, Context_v]")
print(f"  Shape: [num_samples, 4, {CONFIG['embedding_dim']}]")

print(f"\nClass distribution:")
print(f"  Train: {train_labels.sum().item():,} positive / {len(train_labels) - train_labels.sum().item():,} negative")
print(f"  Val: {val_labels.sum().item():,} positive / {len(val_labels) - val_labels.sum().item():,} negative")
print(f"  Test: {test_labels.sum().item():,} positive / {len(test_labels) - test_labels.sum().item():,} negative")

## 9. Save Processed Data

In [ ]:
print("="*70)
print(" "*20 + "STEP 6: Saving Processed Data")
print("="*70)

output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}\n")

# Save train data
train_path = output_dir / "train_data.pt"
torch.save({
    'sequences': train_sequences,
    'labels': train_labels
}, train_path)
train_size = os.path.getsize(train_path) / (1024**2)
print(f"✓ Saved: {train_path.name}")
print(f"  Sequences: {train_sequences.shape}")
print(f"  Labels: {train_labels.shape}")
print(f"  Size: {train_size:.2f} MB")

# Save validation data
val_path = output_dir / "val_data.pt"
torch.save({
    'sequences': val_sequences,
    'labels': val_labels
}, val_path)
val_size = os.path.getsize(val_path) / (1024**2)
print(f"\n✓ Saved: {val_path.name}")
print(f"  Sequences: {val_sequences.shape}")
print(f"  Labels: {val_labels.shape}")
print(f"  Size: {val_size:.2f} MB")

# Save test data
test_path = output_dir / "test_data.pt"
torch.save({
    'sequences': test_sequences,
    'labels': test_labels
}, test_path)
test_size = os.path.getsize(test_path) / (1024**2)
print(f"\n✓ Saved: {test_path.name}")
print(f"  Sequences: {test_sequences.shape}")
print(f"  Labels: {test_labels.shape}")
print(f"  Size: {test_size:.2f} MB")

# Save metadata
metadata = {
    'dataset': 'Pubmed',
    'num_nodes': int(data.num_nodes),
    'num_edges': int(num_edges),
    'embedding_dim': CONFIG['embedding_dim'],
    'encoder_model': CONFIG['encoder_model'],
    'negative_sampling': {
        'ratio': CONFIG['neg_sampling_ratio'],
        'hard_ratio': CONFIG['hard_negative_ratio'],
        'strategy': f"{CONFIG['hard_negative_ratio']*100:.0f}% hard + {(1-CONFIG['hard_negative_ratio'])*100:.0f}% random"
    },
    'splits': {
        'train': {'positive': int(len(train_edges)), 'negative': int(len(train_neg_edges))},
        'val': {'positive': int(len(val_edges)), 'negative': int(len(val_neg_edges))},
        'test': {'positive': int(len(test_edges)), 'negative': int(len(test_neg_edges))}
    }
}

metadata_path = output_dir / "metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
metadata_size = os.path.getsize(metadata_path) / 1024
print(f"\n✓ Saved: {metadata_path.name}")
print(f"  Size: {metadata_size:.2f} KB")

# Calculate total size
total_size = train_size + val_size + test_size + (metadata_size / 1024)

print(f"\n{'='*70}")
print(f"✅ ALL FILES SAVED SUCCESSFULLY!")
print(f"{'='*70}")
print(f"Total size: {total_size:.2f} MB")
print(f"Location: {output_dir}")

## 10. Summary and Verification

In [ ]:
print("="*70)
print(" "*20 + "✅ PROCESSING COMPLETE!")
print("="*70)

print("\n📊 Summary:")
print(f"  • Dataset: Pubmed Citation Network")
print(f"  • Task: Link Prediction for GWM-RNN")
print(f"  • Nodes: {data.num_nodes:,}")
print(f"  • Edges: {num_edges:,}")
print(f"  • Encoder: {CONFIG['encoder_model']}")
print(f"  • Embedding dimension: {CONFIG['embedding_dim']}")

print(f"\n🔢 Data Splits:")
print(f"  • Train: {len(train_sequences):,} samples ({len(train_edges):,} pos + {len(train_neg_edges):,} neg)")
print(f"  • Val: {len(val_sequences):,} samples ({len(val_edges):,} pos + {len(val_neg_edges):,} neg)")
print(f"  • Test: {len(test_sequences):,} samples ({len(test_edges):,} pos + {len(test_neg_edges):,} neg)")

print(f"\n⚖️  Negative Sampling:")
print(f"  • Strategy: {CONFIG['hard_negative_ratio']*100:.0f}% hard (2-hop) + {(1-CONFIG['hard_negative_ratio'])*100:.0f}% random")
print(f"  • Expected: Balanced recall/precision, fewer false negatives")

print(f"\n📦 Sequence Format:")
print(f"  • Structure: [Self_u, Context_u, Self_v, Context_v]")
print(f"  • Shape: [num_samples, 4, {CONFIG['embedding_dim']}]")
print(f"  • Labels: 1 (positive link), 0 (negative link)")

print(f"\n📥 Files Generated:")
for file in sorted(output_dir.glob("*")):
    size = os.path.getsize(file)
    if file.suffix == '.pt':
        size_str = f"{size / (1024**2):.2f} MB"
    else:
        size_str = f"{size / 1024:.2f} KB"
    print(f"  • {file.name} ({size_str})")

print(f"\n🚀 Next Steps:")
print(f"  1. Download files from Kaggle output")
print(f"  2. Upload to training environment")
print(f"  3. Train GWM-RNN model:")
print(f"     python train.py \\")
print(f"       --data_dir {output_dir.name} \\")
print(f"       --hidden_dim 256 \\")
print(f"       --batch_size 512 \\")
print(f"       --learning_rate 1e-3")

print(f"\n{'='*70}")
print("Notebook completed successfully! 🎉")
print(f"{'='*70}")

## Optional: Verify Loaded Data

In [ ]:
# Quick verification - load and check data
print("Verifying saved data...\n")

# Load train data
loaded_train = torch.load(train_path)
print(f"✓ Train data loaded:")
print(f"  Sequences: {loaded_train['sequences'].shape}")
print(f"  Labels: {loaded_train['labels'].shape}")
assert loaded_train['sequences'].shape == train_sequences.shape
assert loaded_train['labels'].shape == train_labels.shape

# Load validation data
loaded_val = torch.load(val_path)
print(f"\n✓ Validation data loaded:")
print(f"  Sequences: {loaded_val['sequences'].shape}")
print(f"  Labels: {loaded_val['labels'].shape}")
assert loaded_val['sequences'].shape == val_sequences.shape
assert loaded_val['labels'].shape == val_labels.shape

# Load test data
loaded_test = torch.load(test_path)
print(f"\n✓ Test data loaded:")
print(f"  Sequences: {loaded_test['sequences'].shape}")
print(f"  Labels: {loaded_test['labels'].shape}")
assert loaded_test['sequences'].shape == test_sequences.shape
assert loaded_test['labels'].shape == test_labels.shape

# Load metadata
with open(metadata_path, 'r') as f:
    loaded_metadata = json.load(f)
print(f"\n✓ Metadata loaded:")
print(json.dumps(loaded_metadata, indent=2))

print("\n✅ All data verified successfully!")

# Show example sequence
print(f"\n📝 Example Sequence (Train):")
print(f"  Shape: {loaded_train['sequences'][0].shape}")
print(f"  Label: {loaded_train['labels'][0].item()} ({'positive link' if loaded_train['labels'][0].item() == 1 else 'negative link'})")
print(f"  Structure:")
print(f"    [0] Self_u:    {loaded_train['sequences'][0][0][:5].numpy()} ... (dim={CONFIG['embedding_dim']})")
print(f"    [1] Context_u: {loaded_train['sequences'][0][1][:5].numpy()} ...")
print(f"    [2] Self_v:    {loaded_train['sequences'][0][2][:5].numpy()} ...")
print(f"    [3] Context_v: {loaded_train['sequences'][0][3][:5].numpy()} ...")